# ⚡ 의도 파악 생성 모델 v2 (nlu-gen-v2, Colab · LoRA) — CPU 를 줄여도 빠르게

v1(Qwen2.5-1.5B)은 정확하지만 CPU 8개가 있어야 문장당 1~2초였습니다. v2 는 세 가지를 바꿔 **CPU 2~4개에서도 빠르게** 만듭니다.

| | v1 | v2 |
|---|---|---|
| 기반 모델 | Qwen2.5-1.5B | **Qwen2.5-0.5B** (약 3배 빠름) |
| 입력 | 지시문 111토큰 + 손님 말 | **`단계|손님 말` 한 줄** (약 15토큰) |
| 답 | "맥스파이시 상하이 버거 2개 주세요" (15토큰) | **`+M02*2`** (약 5토큰) → 서버가 표준 문장으로 바꿈 |

| 단계 | 하는 일 | 시간 (A100 / L4) |
|---|---|---|
| 0 | GPU 확인 · 설치 | 2분 |
| 1 | 데이터 가져오기 (`nlu_data.zip`, 새로 만든 것) | 1분 |
| 2 | 학습 전 실력 (기준점) | 1분 |
| 3 | LoRA 학습 (3 epochs) | 3분 / 6분 |
| 4 | 처음 보는 말 시험 (148문장) | 1분 |
| 5 | 합치기 · GGUF 변환 (Q8_0) · CPU 속도 재기 | 5분 |
| 6 | Drive 저장 | 1분 |

**사용법:** `런타임 → 런타임 유형 변경 → A100 GPU`(없으면 L4) → 위에서부터 셀을 하나씩 실행(▶).
1단계에서 Drive 연결을 허용하면 PC 와 동기화된 `다른 컴퓨터/…/프로젝트1/colab` 에서 데이터를 가져오고 결과도 거기에 저장합니다.

## 0. GPU 확인 · 라이브러리 설치

In [ ]:
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "accelerate", "gguf", "sentencepiece"], check=True)
    # Colab 에 깔린 torchao(0.10)는 최신 peft 와 안 맞아 LoRA 만들 때 ImportError → 쓰지 않으므로 지운다
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 (CPU)")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# bf16 이 되는 GPU(A100·L4)는 bf16, 안 되는 GPU(T4)·CPU 는 float32 (0.5B 라 float32 도 충분히 들어감)
DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
print("dtype:", DTYPE)

## 1. 데이터 가져오기

In [ ]:
import os, zipfile, shutil, glob, csv, json, random
WORK = "/content/nlu_gen2" if IN_COLAB else os.environ.get("NLU_WORK", "./nlu_gen2_work")
os.makedirs(WORK, exist_ok=True)
ZIP = os.path.join(WORK, "nlu_data.zip")

def find_drive_dir():
    cands = glob.glob("/content/drive/Othercomputers/*/프로젝트1/colab") + ["/content/drive/MyDrive/fundamental1"]
    for d in cands:
        if os.path.exists(os.path.join(d, "nlu_data.zip")):
            return d
    return None

DRIVE_DIR = None
if IN_COLAB:
    from google.colab import drive, files
    try:
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive 연결 안 함:", e)
    DRIVE_DIR = find_drive_dir()
    if DRIVE_DIR:
        shutil.copy(os.path.join(DRIVE_DIR, "nlu_data.zip"), ZIP)
        print("Drive 에서 가져옴:", DRIVE_DIR)
    else:
        print("nlu_data.zip 을 골라 주세요")
        up = files.upload()
        shutil.move(next(iter(up)), ZIP)
else:
    shutil.copy(os.environ["NLU_DATA_ZIP"], ZIP)
with zipfile.ZipFile(ZIP) as z:
    z.extractall(WORK)
DATA = os.path.join(WORK, "nlu")
assert os.path.exists(os.path.join(DATA, "gen2_train.jsonl")), "v2 데이터가 없는 예전 nlu_data.zip 입니다. 새 파일로 바꿔 주세요"

read = lambda name: [json.loads(l) for l in open(os.path.join(DATA, name), encoding="utf-8")]
train, val, test = read("gen2_train.jsonl"), read("gen2_val.jsonl"), read("gen2_testset.jsonl")
meta = json.load(open(os.path.join(DATA, "gen2_meta.json"), encoding="utf-8"))
if os.environ.get("NLU_SMOKE"):
    train, val, test = train[:64], val[:16], test[:8]
print(f"학습 {len(train)} · 검증 {len(val)} · 시험 {len(test)}")
print("메뉴 코드:", ", ".join(f"{k}={v}" for k, v in list(meta["menus"].items())[:6]), "…")
for r in random.Random(0).sample(train, 6):
    print(f"  {r['step']}|{r['text']}  →  {r['code']}")

## 2. 학습 전 실력 (기준점)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

BASE = os.environ.get("NLU_GEN_BASE", "Qwen/Qwen2.5-0.5B")  # 지시문 없이 쓰므로 Instruct 아닌 기본 모델
tok = AutoTokenizer.from_pretrained(BASE)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
EOS = tok.eos_token  # 답 끝 표시
model = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=DTYPE).to(DEVICE)

def prompt_text(step, text):
    return meta["prompt_format"].format(step=step, text=text)  # "MENU|그 매운 닭 버거 두 개\n"

@torch.no_grad()
def generate(rows, bs=int(os.environ.get("NLU_GEN_BS", 64))):
    model.eval()
    tok.padding_side = "left"
    outs = []
    for i in range(0, len(rows), bs):
        b = rows[i:i + bs]
        enc = tok([prompt_text(r["step"], r["text"]) for r in b], return_tensors="pt", padding=True).to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=12, do_sample=False, pad_token_id=tok.pad_token_id,
                             eos_token_id=[tok.eos_token_id, tok.convert_tokens_to_ids("Ċ")])
        outs += [s.split("\n")[0].strip() for s in tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)]
    return outs

def evaluate(rows, show=0):
    outs = generate(rows)
    errors = [(r["step"], r["text"], o, r["code"]) for r, o in zip(rows, outs) if o.replace(" ", "") != r["code"].replace(" ", "")]
    for e in errors[:show]:
        print(f"  [{e[0]}] {e[1]} → {e[2]} / 정답 {e[3]}")
    return 1 - len(errors) / len(rows), errors

acc0, _ = evaluate(test, show=3)
print(f"학습 전 정확도: {acc0:.1%}")

## 3. LoRA 학습

In [ ]:
import time
from peft import LoraConfig, get_peft_model
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup

EPOCHS = int(os.environ.get("NLU_EPOCHS", 3))
BATCH = int(os.environ.get("NLU_BATCH", 32))
LR = 3e-4
MAX_STEPS = int(os.environ.get("NLU_MAX_STEPS", 0))
torch.manual_seed(0)

model = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
model.print_trainable_parameters()

def encode(r):
    p = tok(prompt_text(r["step"], r["text"]), add_special_tokens=False)["input_ids"]
    a = tok(r["code"] + "\n", add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    return p + a, [-100] * len(p) + a  # 답 부분만 배운다

def collate(b):
    pairs = [encode(r) for r in b]
    n = max(len(x) for x, _ in pairs)
    ids = torch.full((len(b), n), tok.pad_token_id)
    labels = torch.full((len(b), n), -100)
    mask = torch.zeros((len(b), n), dtype=torch.long)
    for i, (x, y) in enumerate(pairs):
        ids[i, :len(x)] = torch.tensor(x); labels[i, :len(y)] = torch.tensor(y); mask[i, :len(x)] = 1
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}

loader = DataLoader(train, batch_size=BATCH, shuffle=True, collate_fn=collate)
opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.0)
total = MAX_STEPS or EPOCHS * len(loader)
sched = get_linear_schedule_with_warmup(opt, int(total * 0.05), total)
step, t0 = 0, time.time()
for epoch in range(EPOCHS):
    model.train()
    for b in loader:
        b = {k: v.to(DEVICE) for k, v in b.items()}
        loss = model(**b).loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad()
        step += 1
        if step % 50 == 0 or step == total:
            print(f"step {step}/{total} · loss {loss.item():.3f} · {time.time() - t0:.0f}초")
        if step >= total:
            break
    vacc, _ = evaluate(val[:300])
    print(f"epoch {epoch + 1}: 검증 정확도 {vacc:.1%}")
    if step >= total:
        break

## 4. 처음 보는 말 시험 (사람이 쓴 148문장, v1 은 95.3%)

In [ ]:
acc1, errors = evaluate(test, show=40)
print(f"\n처음 보는 말 정확도: 학습 전 {acc0:.1%} → 학습 후 {acc1:.1%}  (틀린 것 {len(errors)}개, 위에 보임)")
print("※ v1(1.5B) 은 95.3%. 많이 낮으면 2단계 BASE 를 'Qwen/Qwen2.5-1.5B' 로 바꿔 다시 돌려도 됩니다 (그래도 짧은 입력·답 덕분에 v1 보다 빠름)")

## 5. 합치기 · GGUF 변환 · CPU 속도 재기
0.5B 는 4비트로 줄이면 정확도가 떨어지기 쉬워 **8비트(Q8_0, 약 0.5GB)** 로 저장합니다. 1.5B 로 학습했으면 Q4_K_M 으로 줄입니다.
마지막에 llama.cpp 로 **CPU 2개·4개 속도**를 재 봅니다.

In [ ]:
merged_dir = os.path.join(WORK, "merged")
model = model.merge_and_unload()
model.save_pretrained(merged_dir, safe_serialization=True)
tok.save_pretrained(merged_dir)

LLAMA = os.path.join(WORK, "llama.cpp")
if not os.path.exists(LLAMA):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA], check=True)
OUT = os.path.join(WORK, "nlu-gen-v2")
os.makedirs(OUT, exist_ok=True)
final = os.path.join(OUT, "model.gguf")
small = "0.5B" in BASE
quant = "Q8_0" if small else "Q4_K_M"
if small:
    subprocess.run([sys.executable, os.path.join(LLAMA, "convert_hf_to_gguf.py"), merged_dir, "--outfile", final,
                    "--outtype", "q8_0"], check=True)
else:
    f16 = os.path.join(WORK, "model-f16.gguf")
    subprocess.run([sys.executable, os.path.join(LLAMA, "convert_hf_to_gguf.py"), merged_dir, "--outfile", f16,
                    "--outtype", "f16"], check=True)
SKIP_BENCH = bool(os.environ.get("NLU_SKIP_BENCH"))  # 로컬 시험용 (Colab 에서는 빌드해서 속도까지 잼)
if not SKIP_BENCH:
  subprocess.run(f"cmake -S {LLAMA} -B {LLAMA}/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && "
               f"cmake --build {LLAMA}/build --target llama-quantize llama-bench -j 8 > /dev/null", shell=True, check=True)
if not small:
    subprocess.run([f"{LLAMA}/build/bin/llama-quantize", f16, final, quant], check=True, stdout=subprocess.DEVNULL)
print(f"{quant} · {os.path.getsize(final) / 1e9:.2f}GB")

# CPU 속도: 입력 16토큰 처리 + 답 6토큰 쓰기 (실제 키오스크 한 문장과 비슷)
for threads in (() if SKIP_BENCH else (2, 4)):
    r = subprocess.run([f"{LLAMA}/build/bin/llama-bench", "-m", final, "-t", str(threads), "-p", "16", "-n", "6", "-o", "json"],
                       capture_output=True, text=True)
    try:
        res = json.loads(r.stdout)
        pp = next(x["avg_ts"] for x in res if x["n_prompt"] > 0)
        tg = next(x["avg_ts"] for x in res if x["n_gen"] > 0)
        print(f"CPU {threads}개: 입력 {pp:.0f}토큰/초 · 답 {tg:.0f}토큰/초 → 한 문장 약 {16 / pp + 6 / tg:.2f}초 (Colab CPU 기준)")
    except Exception:
        print(f"CPU {threads}개 속도 측정 실패:", r.stderr[-300:])

## 6. 저장

In [ ]:
info = {**meta, "model_version": "nlu-gen-v2", "base_model": BASE, "quant": quant,
        "metrics": {"test_acc_before": round(acc0, 4), "test_acc": round(acc1, 4), "test_size": len(test)},
        "trained_at": time.strftime("%Y-%m-%d %H:%M")}
json.dump(info, open(os.path.join(OUT, "meta.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=2)
print(json.dumps(info["metrics"], ensure_ascii=False))
result = os.path.join(WORK, "nlu-gen-v2.zip")
with zipfile.ZipFile(result, "w", zipfile.ZIP_STORED) as z:
    for name in sorted(os.listdir(OUT)):
        z.write(os.path.join(OUT, name), f"nlu-gen-v2/{name}")
print("결과:", result, f"{os.path.getsize(result) / 1e9:.2f}GB")
if IN_COLAB:
    save_dir = DRIVE_DIR or (find_drive_dir() if os.path.isdir("/content/drive") else None)
    if save_dir:
        shutil.copy(result, os.path.join(save_dir, "nlu-gen-v2.zip"))
        print("Drive 에 저장:", os.path.join(save_dir, "nlu-gen-v2.zip"), "→ PC 의 프로젝트1/colab 폴더에 자동으로 생깁니다")
    else:
        files.download(result)

## 다 했으면
`nlu-gen-v2.zip` 이 PC 의 `프로젝트1\colab` 에 생기면 Claude 에게 "v2 학습 끝났어" 라고 알려 주세요.
4단계 정확도와 5단계 CPU 속도를 보고, 서버에 붙여 CPU 를 줄일지 정합니다.